# Per-paper results table (GT vs methods)

Given a `paper_id` (Study#), this notebook:

- Finds the latest prediction CSV for each method (`direct_llm`, `static_workflow`, `mas`) for the configured model/tag.
- Loads ground truth rows from `data/wopke_100/annotation/wopke100.xlsx` sheet `labels`.
- Prints a **field-by-field** table showing extracted values from each method + ground truth + presence/confusion + similarity score.


In [1]:
from __future__ import annotations

import csv
import re
import zipfile
from pathlib import Path
import xml.etree.ElementTree as ET

try:
    import pandas as pd  # type: ignore
except Exception:  # pragma: no cover
    pd = None


def find_repo_root(start: Path | None = None) -> Path:
    p = start or Path.cwd()
    for _ in range(6):
        if (p / "outputs").exists() and (p / "src").exists():
            return p
        p = p.parent
    return start or Path.cwd()


repo_root = find_repo_root()
outputs_root = repo_root / "outputs"
gt_xlsx_path = repo_root / "data" / "wopke_100" / "annotation" / "wopke100.xlsx"

# --- Configure which run outputs to load ---
provider_model_tag = "google_gemini-3-1-flash-lite-preview"
n_fields_tag = "42fields"

METHODS: dict[str, str] = {
    "direct_llm": "direct_llm",
    "workflow": "static_workflow",
    "MAS": "mas",
}

print("repo_root:", repo_root)
print("outputs_root:", outputs_root)
print("gt_xlsx_path:", gt_xlsx_path)
print("provider_model_tag:", provider_model_tag)
print("n_fields_tag:", n_fields_tag)
print("methods:", METHODS)


repo_root: /home/com3dian/Github/meta_analysis_agents
outputs_root: /home/com3dian/Github/meta_analysis_agents/outputs
gt_xlsx_path: /home/com3dian/Github/meta_analysis_agents/data/wopke_100/annotation/wopke100.xlsx
provider_model_tag: google_gemini-3-1-flash-lite-preview
n_fields_tag: 42fields
methods: {'direct_llm': 'direct_llm', 'workflow': 'static_workflow', 'MAS': 'mas'}


In [2]:
def is_present(v) -> bool:
    if v is None:
        return False
    s = str(v).strip()
    if not s:
        return False
    s_low = s.lower()
    return s_low not in {"nan", "none", "null", "n/a", "na"}


def read_prediction_csv(csv_path: str) -> tuple[list[dict], list[str]]:
    with open(csv_path, "r", encoding="utf-8", errors="ignore", newline="") as f:
        reader = csv.DictReader(f)
        fieldnames = list(reader.fieldnames or [])
        rows = [dict(r) for r in reader]
    return rows, fieldnames


def scan_latest_for_method(method_prefix: str) -> dict[int, str]:
    """Return {paper_id: csv_path} for latest run per paper_id."""
    if not outputs_root.exists():
        raise FileNotFoundError(f"outputs folder not found: {outputs_root}")

    needle = f"{method_prefix}_{provider_model_tag}_{n_fields_tag}_"
    paper_id_re = re.compile(r"^(?P<paper_id>\d+)_")
    date_re = re.compile(r"_(?P<date>\d{4}-\d{2}-\d{2})\.csv$")

    latest_by_pid: dict[int, dict] = {}
    for dated_dir in sorted(outputs_root.iterdir()):
        if not dated_dir.is_dir():
            continue
        if not re.match(r"^\d{4}-\d{2}-\d{2}$", dated_dir.name):
            continue

        for csv_path in dated_dir.glob("*.csv"):
            name = csv_path.name
            if needle not in name:
                continue

            m_pid = paper_id_re.match(name)
            m_date = date_re.search(name)
            if not m_pid or not m_date:
                continue

            pid = int(m_pid.group("paper_id"))
            date = m_date.group("date")

            prev = latest_by_pid.get(pid)
            if prev is None or date > prev["date"]:
                latest_by_pid[pid] = {"date": date, "csv_path": str(csv_path)}

    return {pid: v["csv_path"] for pid, v in latest_by_pid.items()}


def load_ground_truth_by_study_id() -> dict[int, list[dict]]:
    """Load GT XLSX as {Study#: [row_dicts]} using pure-Python XLSX parsing."""
    if not gt_xlsx_path.exists():
        raise FileNotFoundError(f"Ground truth Excel not found: {gt_xlsx_path}")

    with zipfile.ZipFile(gt_xlsx_path, "r") as zf:
        wb_xml = ET.fromstring(zf.read("xl/workbook.xml"))
        rels_xml = ET.fromstring(zf.read("xl/_rels/workbook.xml.rels"))

        rid = None
        for sh in wb_xml.iter():
            if sh.tag.endswith("sheet") and sh.attrib.get("name") == "labels":
                rid = sh.attrib.get("{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id")
                break
        if rid is None:
            raise ValueError("Could not find GT sheet 'labels'")

        target = None
        for rel in rels_xml.iter():
            if rel.tag.endswith("Relationship") and rel.attrib.get("Id") == rid:
                target = rel.attrib.get("Target")
                break
        if target is None:
            raise ValueError(f"Could not find GT relationship for rid={rid}")

        sheet_xml = ET.fromstring(zf.read("xl/" + target))

        shared_strings: list[str] = []
        if "xl/sharedStrings.xml" in zf.namelist():
            ss_root = ET.fromstring(zf.read("xl/sharedStrings.xml"))
            for si in ss_root.findall(".//{*}si"):
                parts = []
                for t in si.findall(".//{*}t"):
                    parts.append(t.text or "")
                shared_strings.append("".join(parts))

        def col_to_index(letters: str) -> int:
            idx = 0
            for ch in letters:
                idx = idx * 26 + (ord(ch) - 64)
            return idx - 1

        def ref_to_col(ref: str) -> int:
            letters = []
            for ch in ref:
                if ch.isalpha():
                    letters.append(ch)
                else:
                    break
            return col_to_index("".join(letters))

        def cell_value(cel):
            t = cel.attrib.get("t")
            v_el = cel.find("{*}v")
            if v_el is None or v_el.text is None:
                return None
            raw = v_el.text
            if t == "s":
                try:
                    return shared_strings[int(raw)]
                except Exception:
                    return raw
            return raw

        # Parse rows
        rows_xml = sheet_xml.findall(".//{*}sheetData/{*}row")
        header_list: list[str] = []
        data_rows: list[dict] = []

        for r_idx, r_el in enumerate(rows_xml):
            c_els = r_el.findall("{*}c")
            max_col = 0
            rec_values: dict[int, object] = {}
            any_non_none = False

            for c_el in c_els:
                ref = c_el.attrib.get("r")
                if not ref:
                    continue
                col = ref_to_col(ref)
                max_col = max(max_col, col)
                val = cell_value(c_el)
                rec_values[col] = val
                if val is not None:
                    any_non_none = True

            if not any_non_none:
                continue

            # First non-empty row becomes header
            if not header_list:
                header_list = [str(rec_values.get(i, "") or "") for i in range(max_col + 1)]

                # Deduplicate header names (mimics the existing notebook)
                seen: dict[str, int] = {}
                deduped: list[str] = []
                for col_name in header_list:
                    col_name = col_name if col_name is not None else ""
                    if col_name in seen:
                        seen[col_name] += 1
                        deduped.append(f"{col_name}.{seen[col_name]}")
                    else:
                        seen[col_name] = 0
                        deduped.append(col_name)
                header_list = deduped
                continue

            rec = {header_list[i]: rec_values.get(i) for i in range(len(header_list))}
            data_rows.append(rec)

        # Group by Study#
        gt_by_id: dict[int, list[dict]] = {}
        for rec in data_rows:
            study_key = "Study#" if "Study#" in rec else next((k for k in rec.keys() if k.startswith("Study#")), None)
            if study_key is None:
                continue
            sid_val = rec.get(study_key)
            if sid_val is None:
                continue

            try:
                sid_int = int(float(sid_val))
            except Exception:
                sid_int = int(str(sid_val).strip().split()[0])

            gt_by_id.setdefault(sid_int, []).append(rec)

        return gt_by_id


# --- Load data ---
pred_by_method: dict[str, dict[int, str]] = {}
for method_label, method_prefix in METHODS.items():
    pred_by_method[method_label] = scan_latest_for_method(method_prefix)
    print(method_label, "papers:", len(pred_by_method[method_label]))

gt_by_id = load_ground_truth_by_study_id()
print("GT papers:", len(gt_by_id))

# Canonical prediction fields + intersection with GT keys
any_csv = None
for m in METHODS.keys():
    if pred_by_method[m]:
        any_csv = next(iter(pred_by_method[m].values()))
        break
if any_csv is None:
    raise SystemExit("No matching prediction CSVs found.")

_, canonical_pred_fields = read_prediction_csv(any_csv)

some_pid = next(iter(gt_by_id.keys()))
some_gt_rows = gt_by_id.get(some_pid, [])
gt_keys = set(some_gt_rows[0].keys()) if some_gt_rows else set()
shared_fields = [f for f in canonical_pred_fields if f in gt_keys]

print("canonical_pred_fields:", len(canonical_pred_fields))
print("shared_fields (intersection with GT header):", len(shared_fields))
print("example shared_fields:", shared_fields[:10])


direct_llm papers: 10
workflow papers: 10
MAS papers: 10
GT papers: 100
canonical_pred_fields: 42
shared_fields (intersection with GT header): 39
example shared_fields: ['Year of data', 'Duration of experiment', 'Experimental design', 'Sowing date 1', 'Sowing date 2', 'Harvest date 1', 'Harvest date 2', 'Lat', 'Lon', 'Crop species 1']


In [3]:
# --- Similarity scoring (ported from the existing notebook) ---
import decimal
from datetime import date, timedelta

_MISSING_TOKENS = {"", "nan", "none", "null", "n/a", "na"}
_YEAR_RE = re.compile(r"\b(1\d{3}|20\d{2})\b")
_PURE_NUMBER_RE = re.compile(r"^[+-]?[\d,]+\.?\d*$")
_SCIENTIFIC_NAME_RE = re.compile(r"\s*\([A-Z][a-z]+(?:\s+[a-z]+\.?)+\)")


def _is_missing(v: object) -> bool:
    if v is None:
        return True
    return str(v).strip().lower() in _MISSING_TOKENS


def _try_parse_decimal(v: object) -> object:
    s = str(v).strip().replace(",", "")
    if not _PURE_NUMBER_RE.match(s):
        return None
    try:
        return decimal.Decimal(s)
    except (decimal.InvalidOperation, ValueError):
        return None


def _remove_decimal_point_numeric_string(v: object) -> str | None:
    s = str(v).strip().replace(",", "")
    if not _PURE_NUMBER_RE.match(s):
        return None
    return s.replace(".", "")


def _excel_serial_to_iso(v: object) -> str | None:
    s = str(v).strip().replace(",", "")
    if not _PURE_NUMBER_RE.match(s):
        return None
    try:
        dec = decimal.Decimal(s)
    except (decimal.InvalidOperation, ValueError):
        return None
    try:
        n = int(dec)
    except (OverflowError, ValueError):
        return None
    if n < 20000 or n > 60000:
        return None
    base = date(1900, 1, 1)
    d = base + timedelta(days=n - 2)
    return d.isoformat()


def _normalize_text(v: object) -> str:
    cleaned = _SCIENTIFIC_NAME_RE.sub("", str(v))
    return " ".join(cleaned.lower().split())


def _soft_tokens_match(t1: str, t2: str) -> bool:
    return t1 == t2 or t1.startswith(t2) or t2.startswith(t1)


def _extract_year(v: object) -> str | None:
    m = _YEAR_RE.search(str(v))
    return m.group(1) if m else None


def field_similarity_score(ref_val: object, hyp_val: object) -> float:
    if _is_missing(ref_val) or _is_missing(hyp_val):
        return 0.0

    # 1) Numeric comparison
    ref_dec = _try_parse_decimal(ref_val)
    hyp_dec = _try_parse_decimal(hyp_val)
    if ref_dec is not None and hyp_dec is not None:
        if ref_dec == hyp_dec:
            return 1.0
        ref_no_dot = _remove_decimal_point_numeric_string(ref_val)
        hyp_no_dot = _remove_decimal_point_numeric_string(hyp_val)
        if ref_no_dot is not None and hyp_no_dot is not None and ref_no_dot == hyp_no_dot:
            return 0.5
        return 0.0

    # 2) Excel serial date comparison
    ref_excel = _excel_serial_to_iso(ref_val)
    hyp_excel = _excel_serial_to_iso(hyp_val)
    if ref_excel is not None or hyp_excel is not None:
        def _normalize_dateish(x: object) -> str | None:
            s = str(x).strip()
            if re.match(r"^\d{4}-\d{2}-\d{2}$", s):
                return s
            return _excel_serial_to_iso(s)

        ref_iso = ref_excel or _normalize_dateish(ref_val)
        hyp_iso = hyp_excel or _normalize_dateish(hyp_val)
        if ref_iso is not None and hyp_iso is not None:
            return 1.0 if ref_iso == hyp_iso else 0.0

    # 3) Year comparison
    ref_year = _extract_year(ref_val)
    hyp_year = _extract_year(hyp_val)
    if ref_year is not None and hyp_year is not None:
        return 1.0 if ref_year == hyp_year else 0.0

    # 4) Text categorical comparison (ROUGE-L F1 via soft LCS)
    ref_tokens = _normalize_text(ref_val).split()
    hyp_tokens = _normalize_text(hyp_val).split()
    if not ref_tokens or not hyp_tokens:
        return 0.0

    m, nl = len(ref_tokens), len(hyp_tokens)
    dp = [[0] * (nl + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, nl + 1):
            if _soft_tokens_match(ref_tokens[i - 1], hyp_tokens[j - 1]):
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])

    lcs = dp[m][nl]
    p = lcs / nl
    r = lcs / m
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)


# --- Overall scoring (ported from the existing notebook) ---

def confusion_counts(pred_rows: list[dict], gt_rows: list[dict], fields: list[str]) -> dict:
    """Field-level TP/FP/FN/TN using greedy row matching + tp_avg_similarity."""
    conf = {"TP": 0, "FP": 0, "FN": 0, "TN": 0}
    tp_similarity_total = 0.0
    tp_similarity_n = 0

    def _normalize_label(v: object) -> str:
        s = str(v or "").strip().lower()
        s = re.sub(r"\s+", " ", s)
        return s

    def _find_crop_swap_pairs(cols: list[str]) -> list[tuple[str, str]]:
        col_set = set(cols)
        pairs: list[tuple[str, str]] = []
        seen: set[str] = set()
        suffixes = [
            (" 1", " 2"),
            ("_1", "_2"),
            (".1", ".2"),
        ]
        for c in cols:
            if c in seen:
                continue
            for suffix1, suffix2 in suffixes:
                if c.endswith(suffix1):
                    partner = c[: -len(suffix1)] + suffix2
                    if partner in col_set:
                        pairs.append((c, partner))
                        seen.add(c)
                        seen.add(partner)
                        break
        return pairs

    def _apply_swaps(row: dict, pairs: list[tuple[str, str]]) -> dict:
        if not pairs:
            return row
        out = dict(row)
        for a, b in pairs:
            out[a], out[b] = out.get(b), out.get(a)
        return out

    def _is_label_field(col: str) -> bool:
        cl = col.lower()
        return any(k in cl for k in ["crop species", "crop", "species", "label"])

    def _field_similarity(col: str, ext_val: object, gt_val: object) -> float:
        if not is_present(gt_val) and not is_present(ext_val):
            return 0.0
        if _is_label_field(col):
            return 1.0 if _normalize_label(ext_val) == _normalize_label(gt_val) else 0.0
        return 1.0 if (is_present(gt_val) and is_present(ext_val)) else 0.0

    swap_pairs = _find_crop_swap_pairs(fields)

    def _pair_scores(ext_row: dict, gt_row: dict) -> tuple[float, bool]:
        # Returns (mean_score, swapped_orientation_used)
        def score_orientation(row_values: dict) -> float:
            if not fields:
                return 0.0
            s = 0.0
            for c in fields:
                s += _field_similarity(c, row_values.get(c), gt_row.get(c))
            return s / len(fields)

        mean_orig = score_orientation(ext_row)
        ext_swapped = _apply_swaps(ext_row, swap_pairs)
        mean_swap = score_orientation(ext_swapped)
        return (mean_swap, True) if mean_swap > mean_orig else (mean_orig, False)

    # Greedy matching over all candidate pairs, sorted by mean similarity.
    candidates: list[tuple[float, int, int, bool]] = []
    for ext_i, ext_row in enumerate(pred_rows):
        for gt_i, gt_row in enumerate(gt_rows):
            mean_score, swapped = _pair_scores(ext_row, gt_row)
            candidates.append((mean_score, ext_i, gt_i, swapped))

    candidates.sort(key=lambda x: x[0], reverse=True)
    used_ext: set[int] = set()
    used_gt: set[int] = set()
    matches: list[tuple[int, int, bool]] = []
    for mean_score, ext_i, gt_i, swapped in candidates:
        if ext_i in used_ext or gt_i in used_gt:
            continue
        used_ext.add(ext_i)
        used_gt.add(gt_i)
        matches.append((ext_i, gt_i, swapped))

    # Matched pairs: compare fields cell-by-cell.
    for ext_i, gt_i, swapped in matches:
        gt_row = gt_rows[gt_i]
        ext_row = pred_rows[ext_i]
        ext_row_adj = _apply_swaps(ext_row, swap_pairs) if swapped else ext_row
        for f in fields:
            gt_p = is_present(gt_row.get(f))
            pred_p = is_present(ext_row_adj.get(f))
            if gt_p and pred_p:
                conf["TP"] += 1
                tp_similarity_total += field_similarity_score(gt_row.get(f), ext_row_adj.get(f))
                tp_similarity_n += 1
            elif gt_p and not pred_p:
                conf["FN"] += 1
            elif (not gt_p) and pred_p:
                conf["FP"] += 1
            else:
                conf["TN"] += 1

    # Unmatched GT rows compare against an empty prediction row.
    for gt_i, gt_row in enumerate(gt_rows):
        if gt_i in used_gt:
            continue
        for f in fields:
            gt_p = is_present(gt_row.get(f))
            if gt_p:
                conf["FN"] += 1
            else:
                conf["TN"] += 1

    # Unmatched extracted rows compare against an empty GT row.
    for ext_i, ext_row in enumerate(pred_rows):
        if ext_i in used_ext:
            continue
        for f in fields:
            pred_p = is_present(ext_row.get(f))
            if pred_p:
                conf["FP"] += 1
            else:
                conf["TN"] += 1

    conf["tp_avg_similarity"] = (tp_similarity_total / tp_similarity_n) if tp_similarity_n else float("nan")
    return conf


def overall_metrics(pred_rows: list[dict], gt_rows: list[dict], fields: list[str]) -> dict:
    conf = confusion_counts(pred_rows, gt_rows, fields)
    TP, FP, FN, TN = conf["TP"], conf["FP"], conf["FN"], conf["TN"]
    precision = TP / (TP + FP) if (TP + FP) else float("nan")
    recall = TP / (TP + FN) if (TP + FN) else float("nan")
    specificity = TN / (TN + FP) if (TN + FP) else float("nan")
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else float("nan")
    return {
        **conf,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
    }


# --- Per-paper table helpers ---

def _pretty_values(rows: list[dict], field: str, *, max_items: int = 6) -> str:
    vals: list[str] = []
    seen = set()
    for r in rows:
        v = r.get(field)
        if not is_present(v):
            continue
        s = str(v).strip()
        if s not in seen:
            seen.add(s)
            vals.append(s)
    if not vals:
        return ""
    if len(vals) <= max_items:
        return "; ".join(vals)
    return "; ".join(vals[:max_items]) + f" …(+{len(vals)-max_items})"


def _best_field_similarity(gt_rows: list[dict], pred_rows: list[dict], field: str) -> float:
    gt_vals = [r.get(field) for r in gt_rows if is_present(r.get(field))]
    pr_vals = [r.get(field) for r in pred_rows if is_present(r.get(field))]
    if not gt_vals or not pr_vals:
        return float("nan")
    best = 0.0
    for gv in gt_vals:
        for pv in pr_vals:
            best = max(best, field_similarity_score(gv, pv))
    return best


def _presence_label(gt_rows: list[dict], pred_rows: list[dict], field: str) -> str:
    gt_p = any(is_present(r.get(field)) for r in gt_rows)
    pr_p = any(is_present(r.get(field)) for r in pred_rows)
    if gt_p and pr_p:
        return "TP"
    if gt_p and (not pr_p):
        return "FN"
    if (not gt_p) and pr_p:
        return "FP"
    return "TN"


def paper_label(pid: int) -> str:
    rows = gt_by_id.get(pid, [])
    if not rows:
        return str(pid)
    rec = rows[0]

    author = rec.get("Author")
    title = rec.get("Title")
    a = author.strip() if isinstance(author, str) else ""
    t = title.strip() if isinstance(title, str) else ""
    if a and t:
        return f"{a} — {t}"
    return a or t or str(pid)


def show_paper(paper_id: int, *, fields: list[str] | None = None):
    gt_rows = gt_by_id.get(paper_id, [])
    if not gt_rows:
        raise KeyError(f"paper_id={paper_id} not found in GT")

    pred_rows_by_method: dict[str, list[dict]] = {}
    for method_label in METHODS.keys():
        csv_path = pred_by_method[method_label].get(paper_id)
        if csv_path is None:
            pred_rows_by_method[method_label] = []
        else:
            pred_rows_by_method[method_label], _ = read_prediction_csv(csv_path)

    use_fields = fields or shared_fields

    print(f"paper_id={paper_id} | {paper_label(paper_id)}")
    print("rows in GT:", len(gt_rows))
    for m in METHODS.keys():
        print(f"rows in {m}:", len(pred_rows_by_method[m]))

    # --- Overall scores ---
    overall_rows = []
    for m in ["direct_llm", "workflow", "MAS"]:
        met = overall_metrics(pred_rows_by_method[m], gt_rows, use_fields)
        overall_rows.append({"method": m, **met})

    if pd is not None:
        overall_df = pd.DataFrame(overall_rows)
        display(overall_df)
    else:
        overall_df = None
        for r in overall_rows:
            print(r)

    # --- Field-by-field table ---
    table_rows = []
    for f in use_fields:
        r = {
            "field": f,
            "gt": _pretty_values(gt_rows, f),
            "direct_llm": _pretty_values(pred_rows_by_method["direct_llm"], f),
            "workflow": _pretty_values(pred_rows_by_method["workflow"], f),
            "MAS": _pretty_values(pred_rows_by_method["MAS"], f),
            "presence_direct_llm": _presence_label(gt_rows, pred_rows_by_method["direct_llm"], f),
            "presence_workflow": _presence_label(gt_rows, pred_rows_by_method["workflow"], f),
            "presence_MAS": _presence_label(gt_rows, pred_rows_by_method["MAS"], f),
            "best_sim_direct_llm": _best_field_similarity(gt_rows, pred_rows_by_method["direct_llm"], f),
            "best_sim_workflow": _best_field_similarity(gt_rows, pred_rows_by_method["workflow"], f),
            "best_sim_MAS": _best_field_similarity(gt_rows, pred_rows_by_method["MAS"], f),
        }
        table_rows.append(r)

    if pd is None:
        return {"overall": overall_rows, "by_field": table_rows}

    df = pd.DataFrame(table_rows)
    cols = [
        "field",
        "gt",
        "direct_llm",
        "workflow",
        "MAS",
        "presence_direct_llm",
        "presence_workflow",
        "presence_MAS",
        "best_sim_direct_llm",
        "best_sim_workflow",
        "best_sim_MAS",
    ]
    df = df[cols]

    # Explicitly display, so you see both tables even if you assign the return value.
    display(df)

    return {"overall": overall_df, "by_field": df}


In [7]:
# Example usage
# Pick one of the paper_ids that exist in the discovered predictions
paper_ids = sorted(set().union(*(set(pred_by_method[m].keys()) for m in METHODS.keys())))
print("available paper_ids (from predictions):", paper_ids)

pid = paper_ids[0] if paper_ids else 1
out = show_paper(pid)

# If pandas is available, display as a dataframe.
out


available paper_ids (from predictions): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
paper_id=1 | Jensen, ES — Grain yield, symbiotic N-2 fixation and interspecific competition for inorganic N in pea-barley intercrops
rows in GT: 8
rows in direct_llm: 8
rows in workflow: 8
rows in MAS: 8


,method,TP,FP,FN,TN,tp_avg_similarity,precision,recall,specificity,f1
0,direct_llm,280,0,32,0,0.541071,1.0,0.897436,NaN,0.945946
1,workflow,280,0,32,0,0.641071,1.0,0.897436,NaN,0.945946
2,MAS,272,0,40,0,0.659926,1.0,0.871795,NaN,0.931507


,field,gt,direct_llm,workflow,MAS,presence_direct_llm,presence_workflow,presence_MAS,best_sim_direct_llm,best_sim_workflow,best_sim_MAS
0,Year of data,1980; 1981; 1982; 1984,1980; 1981; 1982; 1984,1980; 1981; 1982; 1984,1980; 1981; 1982; 1984,TP,TP,TP,1.0,1.0,1.0
1,Duration of experiment,1,1 growing season,4-yr,,TP,TP,FN,0.5,0.0,NaN
2,Experimental design,Split-plot,Randomized split-plot design,randomized split-plot designs,randomized split-plot designs,TP,TP,TP,0.5,0.5,0.5
3,Sowing date 1,29327; 29689; 30048; 31518,16 April 1980; 13 April 1981; 7 April 1982; 16...,16 April 1980; 13 April 1981; 7 April 1982; 16...,16 April 1980; 13 April 1981; 7 April 1982; 16...,TP,TP,TP,0.0,0.0,0.0
4,Sowing date 2,29327; 29689; 30048; 31518,16 April 1980; 13 April 1981; 7 April 1982; 16...,16 April 1980; 13 April 1981; 7 April 1982; 16...,16 April 1980; 13 April 1981; 7 April 1982; 16...,TP,TP,TP,0.0,0.0,0.0
5,Harvest date 1,13-15 weeks later; 15 weeks later,,,,FN,FN,FN,NaN,NaN,NaN
6,Harvest date 2,13-15 weeks later; 15 weeks later,,,,FN,FN,FN,NaN,NaN,NaN
7,Lat,55.68,55.41,55,55.68,TP,TP,TP,0.0,0.0,1.0
8,Lon,12.08,12.05,12,12.08,TP,TP,TP,0.0,0.0,1.0
9,Crop species 1,Barley,Pisum sativum,Pisum sativum L.,Pisum sativum,TP,TP,TP,0.0,0.0,0.0


{'overall':        method   TP  FP  FN  TN  tp_avg_similarity  precision    recall  \
 0  direct_llm  280   0  32   0           0.541071        1.0  0.897436   
 1    workflow  280   0  32   0           0.641071        1.0  0.897436   
 2         MAS  272   0  40   0           0.659926        1.0  0.871795   
 
    specificity        f1  
 0          NaN  0.945946  
 1          NaN  0.945946  
 2          NaN  0.931507  ,
 'by_field':                      field                                                 gt  \
 0             Year of data                             1980; 1981; 1982; 1984   
 1   Duration of experiment                                                  1   
 2      Experimental design                                         Split-plot   
 3            Sowing date 1                         29327; 29689; 30048; 31518   
 4            Sowing date 2                         29327; 29689; 30048; 31518   
 5           Harvest date 1                  13-15 weeks later; 15 wee

In [5]:
paper_ids

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Evaluation
We assess extraction quality at the cell level by comparing each predicted field with its corresponding ground-truth field, after greedily matching rows within each paper. Each cell pair is categorized into one of four outcomes: True Positive (TP), where both prediction and ground truth contain a value; False Negative (FN), where a ground-truth value is missed by the prediction; False Positive (FP), where a value is predicted despite no ground-truth value (i.e., hallucination); and True Negative (TN), where both are empty. Based on these definitions, we report three complementary metrics. First, Accuracy in True Positive pairs evaluates the correctness of extracted values when both prediction and ground truth are present by computing a normalized, type-aware similarity score in [0,1]  (e.g., accounting for numeric or year formats), averaged across all TP cells. Second, the Factual rate (TP/(TP+FP)) measures precision by quantifying the proportion of predicted values that correspond to actual ground-truth entries, thus reflecting resistance to hallucination. Third, the Retrieval rate (TP/(TP+FN)) captures recall by indicating how many ground-truth values are successfully retrieved by the method. Together, these metrics provide a balanced evaluation of correctness, faithfulness, and coverage. 

—--------------------------
Dates → figure out how to validate. 
Name: “TP Accuracy” instead of similarity 
No specificity
When answer gives multiple options, including 1 correct, the score should be the fraction that is correct. 
Semantic differences are not scored properly (Mg/ha vs. Mg ha-1). → update prompt. 
Experimental design → limited set of options, hard-code some rules. For each option, list different ways of writing it and take max score of each. 
Lon/lat: leave out extra decimals (beyond ground-truth).  Update prompt to prevent rounding down. 
Crop species 1/2: create a copy and swap the 1 and 2 labels. Report max score across both.
Yield unit: have function that takes GT and replaces ‘/’ with space and add ‘-1’ at the end. report max score of both. Also swap Mg vs t; both are equivalent. 
Species name: double-check that GT is always common name, and if so, update prompt .

paper_id=10; row 0 has 1.0 score for best_workflow which should be < 1
same issue paper_id=0 for N input. All scores only have 1-decimal float .



To ask Wopke:
paper_id=0: unit is different (between original paper and Wopke’s dataset), and therefore also yield values. Why?

Idea:
Use rules for validation, hard-code specific rules for these columns. 
At the very end, use LLM as a judge to compare accuracy. If accurate: LLM as a judge can be default, and rule-based verifies it. If inaccurate, nice extra result (and rule-based stays). 


